In [1]:
import numpy as np
import pandas as pd
import warnings 
warnings.filterwarnings('ignore')

#### Step-1: Loading DataSet

In [2]:
df=pd.read_csv("Data\Food_Delivery_Times.csv")
df.head()

,Order_ID,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
0,522,7.93,Windy,Low,Afternoon,Scooter,12,1.0,43
1,738,16.42,Clear,Medium,Evening,Bike,20,2.0,84
2,741,9.52,Foggy,Low,Night,Scooter,28,1.0,59
3,661,7.44,Rainy,Medium,Afternoon,Scooter,5,1.0,37
4,412,19.03,Clear,Low,Morning,Bike,16,5.0,68


#### Step-2: Understanding DataSet

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Order_ID                1000 non-null   int64  
 1   Distance_km             1000 non-null   float64
 2   Weather                 970 non-null    str    
 3   Traffic_Level           970 non-null    str    
 4   Time_of_Day             970 non-null    str    
 5   Vehicle_Type            1000 non-null   str    
 6   Preparation_Time_min    1000 non-null   int64  
 7   Courier_Experience_yrs  970 non-null    float64
 8   Delivery_Time_min       1000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 91.3 KB


In [4]:
df.dtypes

Order_ID                    int64
Distance_km               float64
Weather                       str
Traffic_Level                 str
Time_of_Day                   str
Vehicle_Type                  str
Preparation_Time_min        int64
Courier_Experience_yrs    float64
Delivery_Time_min           int64
dtype: object

In [5]:
df.isna().sum()

Order_ID                   0
Distance_km                0
Weather                   30
Traffic_Level             30
Time_of_Day               30
Vehicle_Type               0
Preparation_Time_min       0
Courier_Experience_yrs    30
Delivery_Time_min          0
dtype: int64

#### Step-3: Handling Imbalanced Data

In [6]:
for i in df.columns:

    if pd.api.types.is_numeric_dtype(df[i]):
        df[i] = df[i].fillna(df[i].median())
    else:
        df[i] = df[i].fillna(df[i].mode()[0])

In [7]:
df.isna().sum()

Order_ID                  0
Distance_km               0
Weather                   0
Traffic_Level             0
Time_of_Day               0
Vehicle_Type              0
Preparation_Time_min      0
Courier_Experience_yrs    0
Delivery_Time_min         0
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df.head()

,Order_ID,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
0,522,7.93,Windy,Low,Afternoon,Scooter,12,1.0,43
1,738,16.42,Clear,Medium,Evening,Bike,20,2.0,84
2,741,9.52,Foggy,Low,Night,Scooter,28,1.0,59
3,661,7.44,Rainy,Medium,Afternoon,Scooter,5,1.0,37
4,412,19.03,Clear,Low,Morning,Bike,16,5.0,68


#### Step-4: Seperating Features & Target Variables

In [10]:
# Separate input features and target variable
X = df.drop(columns=["Order_ID", "Delivery_Time_min"])
y = df["Delivery_Time_min"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (1000, 7)
Target shape: (1000,)


#### Step-5: Encoding For Categorical Data

In [11]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

# Select categorical columns
cat_cols = X.select_dtypes(include="object").columns

# Create encoder
encoder = OneHotEncoder(
    drop="first",
    handle_unknown="ignore",
    sparse_output=False
)

# Encode categorical columns
encoded_array = encoder.fit_transform(X[cat_cols])

# Convert encoded data into a DataFrame
encoded_df = pd.DataFrame(
    encoded_array,
    columns=encoder.get_feature_names_out(cat_cols),
    index=X.index
)

# Combine numerical and encoded columns
X_encoded = pd.concat(
    [X.drop(columns=cat_cols), encoded_df],
    axis=1
)

# Display results
print("Before encoding:", X.shape)
print("After encoding:", X_encoded.shape)

X_encoded.head()

Before encoding: (1000, 7)
After encoding: (1000, 14)


,Distance_km,Preparation_Time_min,Courier_Experience_yrs,Weather_Foggy,Weather_Rainy,Weather_Snowy,Weather_Windy,Traffic_Level_Low,Traffic_Level_Medium,Time_of_Day_Evening,Time_of_Day_Morning,Time_of_Day_Night,Vehicle_Type_Car,Vehicle_Type_Scooter
0,7.93,12,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
1,16.42,20,2.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
2,9.52,28,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
3,7.44,5,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,19.03,16,5.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0


#### Step-6: Traing & Testing

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (800, 14)
X_test: (200, 14)
y_train: (800,)
y_test: (200,)


#### Step-7: Normalization for Numerical Data

In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training data shape:", X_train_scaled.shape)
print("Testing data shape:", X_test_scaled.shape)

Training data shape: (800, 14)
Testing data shape: (200, 14)


#### Step-8: Model Building

In [14]:
from sklearn.linear_model import LinearRegression

# Initialize the model
model = LinearRegression()

# Train the model
model.fit(X_train_scaled, y_train)

print("Linear Regression model trained successfully!")

y_pred = model.predict(X_test_scaled)

print("Actual values:", y_test.head().tolist())
print("Predicted values:", y_pred[:5].round(2))

Linear Regression model trained successfully!
Actual values: [32, 68, 39, 44, 85]
Predicted values: [35.27 66.51 44.64 43.88 80.02]


#### Step-9: Model Evaluation

In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE:", round(mae, 2))
print("MSE:", round(mse, 2))
print("RMSE:", round(rmse, 2))
print("R2 Score:", round(r2, 4))

MAE: 5.9
MSE: 77.91
RMSE: 8.83
R2 Score: 0.8262


#### Step-10 Comparing Linear Regression with Ridge & Lasso

In [16]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

models = {
    "Linear Regression": model,
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.01)
}

for name, reg in models.items():
    if name != "Linear Regression":
        reg.fit(X_train_scaled, y_train)

    pred = reg.predict(X_test_scaled)

    print(f"\n{name}")
    print("MAE:", round(mean_absolute_error(y_test, pred), 2))
    print("MSE:", round(mean_squared_error(y_test, pred), 2))
    print("RMSE:", round(np.sqrt(mean_squared_error(y_test, pred)), 2))
    print("R2 Score:", round(r2_score(y_test, pred), 4))


Linear Regression
MAE: 5.9
MSE: 77.91
RMSE: 8.83
R2 Score: 0.8262

Ridge Regression
MAE: 5.9
MSE: 77.95
RMSE: 8.83
R2 Score: 0.8261

Lasso Regression
MAE: 5.91
MSE: 78.02
RMSE: 8.83
R2 Score: 0.8259


#### <b>Cross Validation</b>
> Cross-validation is a model evaluation technique that divides the training data into multiple folds to evaluate how consistently a machine learning model performs on different subsets of the data.

In [27]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge, Lasso

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# 1. Split data
X = df.drop(columns=["Order_ID", "Delivery_Time_min"])
y = df["Delivery_Time_min"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# 2. Preprocessing
num_cols = X.select_dtypes(include="number").columns
cat_cols = X.select_dtypes(include="object").columns

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_cols),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore"))
    ]), cat_cols)
])


# 3. Define models and parameters
models = {
    "Linear Regression": (
        LinearRegression(), {}
    ),

    "Ridge": (
        Ridge(), {"model__alpha": [0.01, 0.1, 1, 10, 100]}
    ),

    "Lasso": (
        Lasso(max_iter=10000),
        {"model__alpha": [0.001, 0.01, 0.1, 1, 10]}
    )
}


# 4. Pipeline + Cross Validation + Hyperparameter Tuning
results = []
best_models = {}

for name, (model, params) in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    grid = GridSearchCV(
        pipeline,
        param_grid=params,
        cv=5,
        scoring="r2",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    best_models[name] = best_model

    results.append({
        "Model": name,
        "Best Parameters": grid.best_params_,
        "CV R2": grid.best_score_,
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mean_squared_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2 Score": r2_score(y_test, y_pred)
    })


# 5. Model Comparison
comparison = pd.DataFrame(results)

comparison = comparison.sort_values(
    by="CV R2", ascending=False
).reset_index(drop=True)

display(comparison.round(4))


# 6. Best Model
best_name = comparison.loc[0, "Model"]
best_model = best_models[best_name]

print("Best Model:", best_name)
print("Best CV R2:", round(comparison.loc[0, "CV R2"], 4))

,Model,Best Parameters,CV R2,MAE,MSE,RMSE,R2 Score
0,Ridge,{'model__alpha': 1},0.7526,5.9132,78.1854,8.8423,0.8256
1,Lasso,{'model__alpha': 0.01},0.7526,5.9160,78.2124,8.8438,0.8255
2,Linear Regression,{},0.7525,5.8992,77.9066,8.8265,0.8262


Best Model: Ridge
Best CV R2: 0.7526


#### Ensemble Algorithms Metrics

In [28]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# 1. Define ensemble models and hyperparameters

ensemble_models = {

    "Random Forest": {
        "model": RandomForestRegressor(random_state=42, n_jobs=-1),
        "params": {
            "model__n_estimators": [100, 200],
            "model__max_depth": [None, 10, 20]
        }
    },

    "Gradient Boosting": {
        "model": GradientBoostingRegressor(random_state=42),
        "params": {
            "model__n_estimators": [100, 200],
            "model__learning_rate": [0.05, 0.1],
            "model__max_depth": [2, 3]
        }
    },

    "XGBoost": {
        "model": XGBRegressor(
            objective="reg:squarederror",
            random_state=42,
            n_jobs=1
        ),
        "params": {
            "model__n_estimators": [100, 200],
            "model__learning_rate": [0.05, 0.1],
            "model__max_depth": [3, 5]
        }
    }
}


# 2. Pipeline + Cross Validation + Hyperparameter Tuning

ensemble_results = []

for name, config in ensemble_models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", config["model"])
    ])

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=config["params"],
        cv=5,
        scoring="r2",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    # Store best model
    best_models[name] = best_model

    # Evaluation
    ensemble_results.append({
        "Model": name,
        "Best Parameters": grid.best_params_,
        "CV R2": grid.best_score_,
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mean_squared_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2 Score": r2_score(y_test, y_pred)
    })

    print(f"{name} completed!")


# 3. Display ensemble comparison

ensemble_comparison = pd.DataFrame(ensemble_results)

ensemble_comparison = ensemble_comparison.sort_values(
    by="CV R2", ascending=False
).reset_index(drop=True)

display(ensemble_comparison.round(4))

Random Forest completed!
Gradient Boosting completed!
XGBoost completed!


,Model,Best Parameters,CV R2,MAE,MSE,RMSE,R2 Score
0,Gradient Boosting,"{'model__learning_rate': 0.1, 'model__max_dept...",0.7274,6.7448,95.7398,9.7847,0.7864
1,XGBoost,"{'model__learning_rate': 0.05, 'model__max_dep...",0.7214,6.6999,94.4798,9.7201,0.7892
2,Random Forest,"{'model__max_depth': None, 'model__n_estimator...",0.6880,7.1369,106.0919,10.3001,0.7633


In [30]:
# Combine regression and ensemble results
final_comparison = pd.concat(
    [comparison, ensemble_comparison],
    ignore_index=True
)

# Sort by CV R2
final_comparison = final_comparison.sort_values(
    by="CV R2",
    ascending=False
).reset_index(drop=True)

# Display final comparison
print("FINAL MODEL COMPARISON")
display(final_comparison.round(4))


# Best model based on Cross-Validation R2
best_cv_name = final_comparison.loc[0, "Model"]
best_cv_model = best_models[best_cv_name]

print("\nBest Model Based on CV R2:", best_cv_name)
print("CV R2:", round(final_comparison.loc[0, "CV R2"], 4))


# Best model based on Test R2 (for reporting)
best_test_index = final_comparison["R2 Score"].idxmax()

best_test_name = final_comparison.loc[best_test_index, "Model"]

print("\nBest Model Based on Test R2:", best_test_name)
print("Test R2:", round(
    final_comparison.loc[best_test_index, "R2 Score"], 4
))


# Save the CV-selected model
import os
import joblib

os.makedirs("models", exist_ok=True)

joblib.dump(
    best_cv_model,
    "models/delivery_time_model.pkl"
)

print("\nBest CV-selected model saved successfully!")

FINAL MODEL COMPARISON


,Model,Best Parameters,CV R2,MAE,MSE,RMSE,R2 Score
0,Ridge,{'model__alpha': 1},0.7526,5.9132,78.1854,8.8423,0.8256
1,Lasso,{'model__alpha': 0.01},0.7526,5.9160,78.2124,8.8438,0.8255
2,Linear Regression,{},0.7525,5.8992,77.9066,8.8265,0.8262
3,Gradient Boosting,"{'model__learning_rate': 0.1, 'model__max_dept...",0.7274,6.7448,95.7398,9.7847,0.7864
4,XGBoost,"{'model__learning_rate': 0.05, 'model__max_dep...",0.7214,6.6999,94.4798,9.7201,0.7892
5,Random Forest,"{'model__max_depth': None, 'model__n_estimator...",0.6880,7.1369,106.0919,10.3001,0.7633



Best Model Based on CV R2: Ridge
CV R2: 0.7526

Best Model Based on Test R2: Linear Regression
Test R2: 0.8262

Best CV-selected model saved successfully!
